# 🚦 Traffic RL – Train on Google Colab

**Steps:**
1. Run cells top-to-bottom (Runtime → Run all)
2. In cell 3, upload your 5 project files when prompted
3. Training runs automatically (~45–90 min on T4 GPU)
4. Download the trained model at the end

> **Tip:** Enable GPU — Runtime → Change runtime type → T4 GPU

## Cell 1 — Install SUMO (headless)

In [ ]:
# Install SUMO via apt (Ubuntu-based Colab runtime)
import subprocess, sys

print('Adding SUMO PPA and installing...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-yqq', 'sumo', 'sumo-tools', 'sumo-doc'], check=True)

import os
os.environ['SUMO_HOME'] = '/usr/share/sumo'
print('SUMO_HOME =', os.environ['SUMO_HOME'])

# Verify
result = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
print(result.stdout.splitlines()[0])

## Cell 2 — Install Python dependencies

In [ ]:
!pip install -q stable-baselines3[extra] gymnasium tensorboard
print('All Python packages installed.')

## Cell 3 — Upload your project files

Upload these **5 files** from your `sumo_test/` folder:
- `cross.sumocfg`
- `cross.net.xml`
- `cross.rou.xml`
- `cross.nod.xml`
- `cross.edg.xml`

In [ ]:
from google.colab import files
import os

WORK_DIR = '/content/sumo_project'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print('Upload your 5 SUMO config files now:')
uploaded = files.upload()   # file picker appears here

print('\nFiles in working directory:')
for f in os.listdir('.'):
    print(' ', f)

## Cell 4 — Write the training script

In [ ]:
script = '''
import os, sys, time, random
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import subprocess
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import torch.nn as nn

os.environ.setdefault("SUMO_HOME", "/usr/share/sumo")
sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))
import traci

ACTION_TO_SUMO_GREEN = {0: 0, 1: 2, 2: 4, 3: 6}
NUM_PHASES = 4

class SumoEnv(gym.Env):
    def __init__(self, sumo_config="cross.sumocfg", gui=False,
                 max_steps=3600, port=None):
        super().__init__()
        self.sumo_binary = "sumo-gui" if gui else "sumo"
        self.sumo_config = sumo_config
        self.gui = gui
        self.port = port or random.randint(9000, 9999)
        self.action_space = spaces.Discrete(NUM_PHASES)
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(17,), dtype=np.float32)
        self.lanes = ["N2J_0", "S2J_0", "E2J_0", "W2J_0"]
        self.tls = "J0"
        self.sim_step = 5
        self.yellow_time = 3
        self.min_green_time = 10
        self.max_green_time = 60
        self.max_steps = max_steps
        self.t = 0
        self.proc = None
        self.current_phase = 0
        self.time_since_last_switch = 0
        self.last_action = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.close()
        cmd = [self.sumo_binary, "-c", self.sumo_config,
               "--no-step-log", "true",
               "--waiting-time-memory", "1000",
               "--time-to-teleport", "-1"]
        self.proc = subprocess.Popen(
            cmd + ["--remote-port", str(self.port)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for attempt in range(10):
            try:
                time.sleep(0.5 + attempt * 0.3)
                traci.init(self.port)
                break
            except Exception:
                continue
        self.t = 0
        self.current_phase = 0
        self.time_since_last_switch = 0
        self.last_action = 0
        traci.trafficlight.setPhase(self.tls, ACTION_TO_SUMO_GREEN[0])
        return self._get_obs(), {}

    def step(self, action):
        action = int(action)
        can_switch = self.time_since_last_switch >= self.min_green_time
        must_switch = self.time_since_last_switch >= self.max_green_time
        switch = False
        if must_switch and action == self.current_phase:
            switch = True
            action = (self.current_phase + 1) % NUM_PHASES
        elif can_switch and action != self.current_phase:
            switch = True
        if switch:
            self._do_yellow_phase()
            self.current_phase = action
            traci.trafficlight.setPhase(self.tls, ACTION_TO_SUMO_GREEN[self.current_phase])
            self.time_since_last_switch = 0
        reward = 0.0
        for _ in range(self.sim_step):
            traci.simulationStep()
            self.t += 1
            self.time_since_last_switch += 1
            reward -= self._calculate_pressure_reward()
        obs = self._get_obs()
        terminated = self.t >= self.max_steps
        self.last_action = action
        return obs, reward, terminated, False, {}

    def _do_yellow_phase(self):
        yellow_sumo_idx = ACTION_TO_SUMO_GREEN[self.current_phase] + 1
        traci.trafficlight.setPhase(self.tls, yellow_sumo_idx)
        for _ in range(self.yellow_time):
            traci.simulationStep()
            self.t += 1

    def _get_obs(self):
        queues, densities, speeds = [], [], []
        for lane in self.lanes:
            queues.append(float(traci.lane.getLastStepHaltingNumber(lane)))
            veh_count = traci.lane.getLastStepVehicleNumber(lane)
            lane_len = traci.lane.getLength(lane)
            densities.append(veh_count / max(lane_len / 5.0, 1.0))
            avg_speed = traci.lane.getLastStepMeanSpeed(lane)
            speed_limit = traci.lane.getMaxSpeed(lane)
            speeds.append(avg_speed / speed_limit if speed_limit > 0 else 0.0)
        phase_oh = [0.0] * NUM_PHASES
        phase_oh[self.current_phase] = 1.0
        time_norm = min(self.time_since_last_switch / self.max_green_time, 1.0)
        return np.concatenate([queues, densities, speeds, phase_oh, [time_norm]]).astype(np.float32)

    def _calculate_pressure_reward(self):
        total_queue, total_wait, max_veh_wait = 0, 0, 0.0
        for lane in self.lanes:
            total_queue += traci.lane.getLastStepHaltingNumber(lane)
            total_wait  += traci.lane.getWaitingTime(lane)
        for veh_id in traci.vehicle.getIDList():
            w = traci.vehicle.getWaitingTime(veh_id)
            if w > max_veh_wait:
                max_veh_wait = w
        return (total_queue + 0.5 * (total_wait / 60.0) + 0.02 * (max_veh_wait / 60.0))

    def close(self):
        try: traci.close()
        except: pass
        if self.proc: self.proc.kill()


def make_env(port=None):
    def _init():
        _port = port or random.randint(9000, 9999)
        return Monitor(SumoEnv(gui=False, max_steps=3600, port=_port))
    return _init

def linear_schedule(initial_lr, final_lr=1e-5):
    def schedule(progress_remaining):
        return final_lr + progress_remaining * (initial_lr - final_lr)
    return schedule


if __name__ == "__main__":
    os.makedirs("./logs/best_model",  exist_ok=True)
    os.makedirs("./logs/checkpoints", exist_ok=True)

    env = DummyVecEnv([make_env()])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    eval_env = DummyVecEnv([make_env()])
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False,
                            clip_obs=10.0, training=False)

    eval_cb = EvalCallback(eval_env,
        best_model_save_path="./logs/best_model",
        log_path="./logs/",
        eval_freq=3000, n_eval_episodes=3,
        deterministic=True, render=False, verbose=1)

    ckpt_cb = CheckpointCallback(
        save_freq=50_000, save_path="./logs/checkpoints/",
        name_prefix="ppo_traffic", verbose=1)

    model = PPO("MlpPolicy", env, verbose=1,
        learning_rate=linear_schedule(3e-4, 1e-5),
        n_steps=4096, batch_size=256, n_epochs=15,
        gamma=0.995, gae_lambda=0.95, clip_range=0.2,
        ent_coef=0.005, vf_coef=0.5, max_grad_norm=0.5,
        policy_kwargs=dict(
            activation_fn=nn.Tanh,
            net_arch=dict(pi=[512,512,256], vf=[512,512,256])),
        tensorboard_log="./traffic_tensorboard/")

    print("Training started...")
    model.learn(total_timesteps=1000_000,
                callback=[eval_cb, ckpt_cb],
                progress_bar=True)

    model.save("ppo_traffic_final")
    env.save("vec_normalize.pkl")
    print("Done. Files saved.")
'''

with open('/content/sumo_project/train.py', 'w') as f:
    f.write(script)
print('train.py written.')

## Cell 5 — (Optional) Mount Google Drive to auto-save results

Skip this if you just want to download at the end.

In [ ]:
# OPTIONAL — mount Google Drive so results survive if Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE = '/content/drive/MyDrive/Traffix_RL'
import os
os.makedirs(DRIVE_SAVE, exist_ok=True)
print('Drive mounted. Results will be mirrored to:', DRIVE_SAVE)

## Cell 6 — Start training

⏱ Estimated time on **T4 GPU**: ~45–90 minutes for 1 000 000 steps.

In [ ]:
import os, time
os.chdir('/content/sumo_project')
os.environ['SUMO_HOME'] = '/usr/share/sumo'

start = time.time()
%run train.py
elapsed = time.time() - start
print(f'\nTraining finished in {elapsed/60:.1f} minutes.')

## Cell 7 — (Optional) Copy results to Google Drive

In [ ]:
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/Traffix_RL'   # adjust if needed

if os.path.isdir(DRIVE_SAVE):
    for fname in ['ppo_traffic_final.zip', 'vec_normalize.pkl']:
        src = f'/content/sumo_project/{fname}'
        if os.path.isfile(src):
            shutil.copy(src, DRIVE_SAVE)
            print(f'Copied {fname} → Drive')

    for folder in ['logs', 'traffic_tensorboard']:
        src = f'/content/sumo_project/{folder}'
        dst = f'{DRIVE_SAVE}/{folder}'
        if os.path.isdir(src):
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'Copied {folder}/ → Drive')
else:
    print('Drive not mounted — skipping (run Cell 5 first).')

## Cell 8 — Download trained model to your PC

In [ ]:
import zipfile, os
from google.colab import files

os.chdir('/content/sumo_project')

# Bundle everything needed to run locally
bundle = 'trained_model_bundle.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['ppo_traffic_final.zip', 'vec_normalize.pkl']:
        if os.path.isfile(fname):
            zf.write(fname)
    # Best model from EvalCallback
    best = 'logs/best_model/best_model.zip'
    if os.path.isfile(best):
        zf.write(best)
    # evaluations.npz for plotting
    npz = 'logs/evaluations.npz'
    if os.path.isfile(npz):
        zf.write(npz)

print('Downloading', bundle, '...')
files.download(bundle)

## Cell 9 — View TensorBoard live (while training or after)

Run this **in a separate cell tab** while training is running in Cell 6.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/sumo_project/traffic_tensorboard